# Example 10.1 / 1: Minimum-cost flow

Companion notebook for **10.1 Network Models** in the AMPL lab Documentation, section "Minimum-cost flow". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Minimum-cost flow

This example routes six units from a plant to a customer through two transfer nodes. In the Python blocks, `ampl` has already read `model.mod`.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NODES;
set ARCS within NODES cross NODES;
param balance {NODES};
param capacity {ARCS} >= 0;
param unit_cost {ARCS} >= 0;

var Flow {(i, j) in ARCS} >= 0, <= capacity[i, j];

minimize TotalCost:
    sum {(i, j) in ARCS} unit_cost[i, j] * Flow[i, j];

subject to FlowBalance {k in NODES}:
    sum {(k, j) in ARCS} Flow[k, j]
  - sum {(i, k) in ARCS} Flow[i, k] = balance[k];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["NODES"] = ["plant", "east", "west", "customer"]
ampl.set["ARCS"] = [("plant", "east"), ("plant", "west"), ("east", "customer"), ("west", "customer")]
ampl.param["balance"] = {"plant": 6, "east": 0, "west": 0, "customer": -6}
ampl.param["capacity"] = {("plant", "east"): 4, ("plant", "west"): 5, ("east", "customer"): 4, ("west", "customer"): 5}
ampl.param["unit_cost"] = {("plant", "east"): 2, ("plant", "west"): 1, ("east", "customer"): 1, ("west", "customer"): 3}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.var["Flow"].to_dict())

HiGHS 1.14.0: optimal solution; objective 20
0 simplex iterations
0 barrier iterations
solved
{('east', 'customer'): 4, ('plant', 'east'): 4, ('plant', 'west'): 2, ('west', 'customer'): 2}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set NODES := plant east west customer;
set ARCS := (plant,east) (plant,west) (east,customer) (west,customer);
param balance := plant 6 east 0 west 0 customer -6;
param capacity := [plant,east] 4 [plant,west] 5 [east,customer] 4 [west,customer] 5;
param unit_cost := [plant,east] 2 [plant,west] 1 [east,customer] 1 [west,customer] 3;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Flow;

HiGHS 1.14.0: optimal solution; objective 20
0 simplex iterations
0 barrier iterations
Flow :=
east  customer   4
plant east       4
plant west       2
west  customer   2
;

